In [ ]:
# EuroSAT Multi-Model Comprehensive Comparison Tool with Gradio UI
# Analyzes 4 models across all 10 categories with detailed metrics

# First, mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully!")
    DRIVE_MOUNTED = True
except ImportError:
    print("⚠️  Not running in Colab - Google Drive mounting not available")
    DRIVE_MOUNTED = False

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import pandas as pd
import requests
from io import BytesIO
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import gradio as gr
import warnings
import os
warnings.filterwarnings('ignore')

# Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

EUROSAT_CLASSES = [
    'AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial',
    'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake'
]

# Default model paths
DEFAULT_MODEL_PATHS = {
    'resnet50': '/content/drive/MyDrive/weights/resnet50_eurosat_best.pth',
    'densenet': '/content/drive/MyDrive/weights/densenet121_eurosat_best.pth',
    'efficientnet': '/content/drive/MyDrive/weights/efficientnet_b3_eurosat_best.pth',
    'vit': '/content/drive/MyDrive/weights/final_eurosat_vit_model.pth',
}

# ==================== Model Architectures ====================

class EuroSATResNet50(nn.Module):
    def __init__(self, num_classes=10, dropout_rate=0.5):
        super(EuroSATResNet50, self).__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate * 0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

class EuroSATDenseNet121(nn.Module):
    def __init__(self, num_classes=10, pretrained=True, dropout_rate=0.4):
        super(EuroSATDenseNet121, self).__init__()
        if pretrained:
            self.backbone = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        else:
            self.backbone = models.densenet121(weights=None)
        num_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate * 0.6),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout_rate * 0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

class EuroSATEfficientNetB3(nn.Module):
    def __init__(self, num_classes=10):
        super(EuroSATEfficientNetB3, self).__init__()
        self.model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
        self.model.classifier[1] = nn.Linear(self.model.classifier[1].in_features, num_classes)

    def forward(self, x):
        return self.model(x)

class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.projection = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        nn.init.xavier_uniform_(self.projection.weight)
        nn.init.zeros_(self.projection.bias)

    def forward(self, x):
        x = self.projection(x)
        x = x.flatten(2).transpose(1, 2)
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim=768, n_heads=12, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(embed_dim, embed_dim * 3, bias=False)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.qkv.weight)
        nn.init.xavier_uniform_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return self.dropout(x)

class MLP(nn.Module):
    def __init__(self, embed_dim=768, hidden_dim=3072, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, embed_dim)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = torch.nn.functional.gelu(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        return self.dropout2(x)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim=768, n_heads=12, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        hidden_dim = int(embed_dim * mlp_ratio)
        self.norm1 = nn.LayerNorm(embed_dim, eps=1e-6)
        self.attn = MultiHeadAttention(embed_dim, n_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim, eps=1e-6)
        self.mlp = MLP(embed_dim, hidden_dim, dropout)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class EuroSATViT(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768,
                 n_layers=8, n_heads=12, mlp_ratio=4.0, n_classes=10, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        n_patches = self.patch_embed.n_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([TransformerBlock(embed_dim, n_heads, mlp_ratio, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(embed_dim, eps=1e-6)
        self.head = nn.Linear(embed_dim, n_classes)
        self.init_weights()

    def init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.xavier_uniform_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        x = x + self.pos_embed
        x = self.dropout(x)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.head(x[:, 0])

# ==================== Utility Functions ====================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def load_image_from_url(url):
    """Load image from URL"""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert('RGB')
        return image
    except Exception as e:
        print(f"Error loading {url}: {e}")
        return None

def predict_image(model, image, model_name):
    """Get predictions from model"""
    if model is None or image is None:
        return None

    try:
        img_tensor = transform(image).unsqueeze(0).to(device)
        with torch.no_grad():
            outputs = model(img_tensor)
            probs = torch.softmax(outputs, dim=1)[0].cpu().numpy()
            pred_idx = np.argmax(probs)
            pred_class = EUROSAT_CLASSES[pred_idx]
            confidence = probs[pred_idx]

        return {
            'model': model_name,
            'predicted_class': pred_class,
            'confidence': confidence,
            'probabilities': probs,
            'all_predictions': dict(zip(EUROSAT_CLASSES, probs))
        }
    except Exception as e:
        print(f"Error predicting with {model_name}: {e}")
        return None

def load_models():
    """Load all models from default paths"""
    models_dict = {}
    status = []

    print("\n" + "="*60)
    print("LOADING MODELS FROM GOOGLE DRIVE")
    print("="*60 + "\n")

    # Load ResNet50
    print(f"Loading ResNet50 from: {DEFAULT_MODEL_PATHS['resnet50']}")
    if os.path.exists(DEFAULT_MODEL_PATHS['resnet50']):
        try:
            checkpoint = torch.load(DEFAULT_MODEL_PATHS['resnet50'], map_location=device)
            model = EuroSATResNet50(num_classes=10)
            if 'model_state_dict' in checkpoint:
                model.load_state_dict(checkpoint['model_state_dict'])
            else:
                model.load_state_dict(checkpoint)
            model.to(device)
            model.eval()
            models_dict['ResNet50'] = model
            status.append("✅ ResNet50 loaded")
            print("✅ ResNet50 loaded successfully\n")
        except Exception as e:
            status.append(f"❌ ResNet50 failed: {str(e)[:50]}")
            print(f"❌ ResNet50 loading failed: {e}\n")
    else:
        status.append(f"❌ ResNet50 file not found")
        print(f"❌ ResNet50 file not found at {DEFAULT_MODEL_PATHS['resnet50']}\n")

    # Load DenseNet-121
    print(f"Loading DenseNet-121 from: {DEFAULT_MODEL_PATHS['densenet']}")
    if os.path.exists(DEFAULT_MODEL_PATHS['densenet']):
        try:
            checkpoint = torch.load(DEFAULT_MODEL_PATHS['densenet'], map_location=device)
            model = EuroSATDenseNet121(num_classes=10)
            if 'model_state_dict' in checkpoint:
                model.load_state_dict(checkpoint['model_state_dict'])
            else:
                model.load_state_dict(checkpoint)
            model.to(device)
            model.eval()
            models_dict['DenseNet-121'] = model
            status.append("✅ DenseNet-121 loaded")
            print("✅ DenseNet-121 loaded successfully\n")
        except Exception as e:
            status.append(f"❌ DenseNet-121 failed: {str(e)[:50]}")
            print(f"❌ DenseNet-121 loading failed: {e}\n")
    else:
        status.append(f"❌ DenseNet-121 file not found")
        print(f"❌ DenseNet-121 file not found at {DEFAULT_MODEL_PATHS['densenet']}\n")

    # Load EfficientNet-B3
    print(f"Loading EfficientNet-B3 from: {DEFAULT_MODEL_PATHS['efficientnet']}")
    if os.path.exists(DEFAULT_MODEL_PATHS['efficientnet']):
        try:
            checkpoint = torch.load(DEFAULT_MODEL_PATHS['efficientnet'], map_location=device)
            model = EuroSATEfficientNetB3(num_classes=10)
            if 'model_state_dict' in checkpoint:
                state_dict = checkpoint['model_state_dict']
            else:
                state_dict = checkpoint
            corrected_state_dict = {f"model.{k}" if k.startswith(('features.', 'classifier.')) else k: v
                                  for k, v in state_dict.items()}
            model.load_state_dict(corrected_state_dict)
            model.to(device)
            model.eval()
            models_dict['EfficientNet-B3'] = model
            status.append("✅ EfficientNet-B3 loaded")
            print("✅ EfficientNet-B3 loaded successfully\n")
        except Exception as e:
            status.append(f"❌ EfficientNet-B3 failed: {str(e)[:50]}")
            print(f"❌ EfficientNet-B3 loading failed: {e}\n")
    else:
        status.append(f"❌ EfficientNet-B3 file not found")
        print(f"❌ EfficientNet-B3 file not found at {DEFAULT_MODEL_PATHS['efficientnet']}\n")

    # Load Vision Transformer
    print(f"Loading Vision Transformer from: {DEFAULT_MODEL_PATHS['vit']}")
    if os.path.exists(DEFAULT_MODEL_PATHS['vit']):
        try:
            checkpoint = torch.load(DEFAULT_MODEL_PATHS['vit'], map_location=device, weights_only=False)
            model = EuroSATViT(n_classes=10)
            if 'model_state_dict' in checkpoint:
                model.load_state_dict(checkpoint['model_state_dict'])
            else:
                model.load_state_dict(checkpoint)
            model.to(device)
            model.eval()
            models_dict['ViT'] = model
            status.append("✅ Vision Transformer loaded")
            print("✅ Vision Transformer loaded successfully\n")
        except Exception as e:
            status.append(f"❌ Vision Transformer failed: {str(e)[:50]}")
            print(f"❌ Vision Transformer loading failed: {e}\n")
    else:
        status.append(f"❌ Vision Transformer file not found")
        print(f"❌ Vision Transformer file not found at {DEFAULT_MODEL_PATHS['vit']}\n")

    print("="*60)
    print(f"✅ Models loaded: {len(models_dict)}/4")
    print("="*60 + "\n")

    return models_dict, "\n".join(status)

def run_comparison(url_annual, url_forest, url_herbaceous, url_highway, url_industrial,
                   url_pasture, url_permanent, url_residential, url_river, url_sealake):
    """Run full comparison on 10 categories"""

    global MODELS_DICT

    if not MODELS_DICT:
        return pd.DataFrame(), pd.DataFrame(), None, "❌ No models loaded"

    image_urls = {
        'AnnualCrop': url_annual,
        'Forest': url_forest,
        'HerbaceousVegetation': url_herbaceous,
        'Highway': url_highway,
        'Industrial': url_industrial,
        'Pasture': url_pasture,
        'PermanentCrop': url_permanent,
        'Residential': url_residential,
        'River': url_river,
        'SeaLake': url_sealake,
    }

    results = []
    all_predictions = {model_name: [] for model_name in MODELS_DICT.keys()}
    all_ground_truth = []

    progress_text = "Processing images...\n"

    for category, url in image_urls.items():
        if not url or not url.strip():
            progress_text += f"⚠️  {category}: URL not provided\n"
            continue

        image = load_image_from_url(url)
        if image is None:
            progress_text += f"❌ {category}: Failed to load image\n"
            continue

        all_ground_truth.append(category)
        row = {'Category': category}

        for model_name, model in MODELS_DICT.items():
            pred = predict_image(model, image, model_name)
            if pred:
                row[f'{model_name}_Pred'] = pred['predicted_class']
                row[f'{model_name}_Conf'] = f"{pred['confidence']:.2%}"
                all_predictions[model_name].append(pred['predicted_class'])

        results.append(row)
        progress_text += f"✅ {category}\n"

    if not results:
        return pd.DataFrame(), pd.DataFrame(), None, "❌ No images processed"

    results_df = pd.DataFrame(results)

    # Calculate accuracy for each model
    accuracy_data = []
    for model_name in MODELS_DICT.keys():
        predictions = all_predictions[model_name]
        if predictions:
            acc = accuracy_score(all_ground_truth, predictions)
            accuracy_data.append({
                'Model': model_name,
                'Accuracy': f'{acc:.2%}',
                'Correct': f"{int(acc * len(all_ground_truth))}/{len(all_ground_truth)}"
            })

    accuracy_df = pd.DataFrame(accuracy_data)

    # Create visualization
    fig = create_visualization_chart(results_df, MODELS_DICT, all_ground_truth)

    progress_text += f"\n{'='*50}\n✅ Analysis Complete!\n"
    for _, row in accuracy_df.iterrows():
        progress_text += f"{row['Model']}: {row['Accuracy']} ({row['Correct']})\n"

    return results_df, accuracy_df, fig, progress_text

def create_visualization_chart(df, models_dict, ground_truth):
    """Create comparison visualization"""
    try:
        model_names = list(models_dict.keys())
        categories = df['Category'].tolist()

        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=('Predictions per Category', 'Model Agreement'),
            specs=[[{"type": "bar"}, {"type": "bar"}]]
        )

        # Plot 1: Correct predictions per category
        for model_name in model_names:
            correct_col = f'{model_name}_Pred'
            if correct_col in df.columns:
                correct = [1 if df.loc[i, correct_col] == categories[i] else 0
                          for i in range(len(df))]
                fig.add_trace(
                    go.Bar(x=categories, y=correct, name=model_name),
                    row=1, col=1
                )

        # Plot 2: Agreement between models
        agreement_scores = []
        for i in range(len(categories)):
            predictions = [df.loc[i, f'{m}_Pred'] for m in model_names if f'{m}_Pred' in df.columns]
            agreement = len(set(predictions)) == 1
            agreement_scores.append(int(agreement))

        fig.add_trace(
            go.Bar(x=categories, y=agreement_scores, name='Full Agreement', marker_color='lightgreen'),
            row=1, col=2
        )

        fig.update_xaxes(title_text="Category", row=1, col=1, tickangle=45)
        fig.update_xaxes(title_text="Category", row=1, col=2, tickangle=45)
        fig.update_yaxes(title_text="Correct (1=Yes, 0=No)", row=1, col=1)
        fig.update_yaxes(title_text="Agreement (1=All Models Agree)", row=1, col=2)
        fig.update_layout(height=500, title_text="Model Performance Comparison", barmode='group')

        return fig
    except Exception as e:
        print(f"Visualization error: {e}")
        return None

# ==================== Load Models at Startup ====================

print("\n🚀 Initializing EuroSAT Multi-Model Comparison Tool...\n")
MODELS_DICT, MODEL_STATUS = load_models()

# ==================== Gradio Interface ====================

def create_ui():
    """Create Gradio interface"""

    with gr.Blocks(theme=gr.themes.Soft(), title="EuroSAT Multi-Model Comparison") as demo:

        gr.Markdown("""
        # 🛰️ EuroSAT Multi-Model Comprehensive Comparison

        Compare predictions from **4 state-of-the-art models** on all **10 EuroSAT categories**.

        **Models:** ResNet50 | DenseNet-121 | EfficientNet-B3 | Vision Transformer

        **Dataset:** 27,000 Sentinel-2 satellite images across 10 land use classes
        """)

        with gr.Row():
            gr.Markdown(f"""
            ### 🔧 Model Status
            ```
            {MODEL_STATUS}
            ```
            """)

        gr.Markdown("### 📷 Enter Image URLs for Each Category")

        with gr.Row():
            url_annual = gr.Textbox(label="AnnualCrop", placeholder="https://example.com/annual_crop.jpg")
            url_forest = gr.Textbox(label="Forest", placeholder="https://example.com/forest.jpg")
            url_herbaceous = gr.Textbox(label="HerbaceousVegetation", placeholder="https://example.com/herbaceous.jpg")

        with gr.Row():
            url_highway = gr.Textbox(label="Highway", placeholder="https://example.com/highway.jpg")
            url_industrial = gr.Textbox(label="Industrial", placeholder="https://example.com/industrial.jpg")
            url_pasture = gr.Textbox(label="Pasture", placeholder="https://example.com/pasture.jpg")

        with gr.Row():
            url_permanent = gr.Textbox(label="PermanentCrop", placeholder="https://example.com/permanent_crop.jpg")
            url_residential = gr.Textbox(label="Residential", placeholder="https://example.com/residential.jpg")
            url_river = gr.Textbox(label="River", placeholder="https://example.com/river.jpg")

        with gr.Row():
            url_sealake = gr.Textbox(label="SeaLake", placeholder="https://example.com/sea_lake.jpg")

        analyze_btn = gr.Button("🔍 Run Comparison", variant="primary", size="lg")

        gr.Markdown("### 📊 Results")

        with gr.Row():
            with gr.Column():
                progress_output = gr.Textbox(label="Progress & Summary", lines=12, max_lines=15)
            with gr.Column():
                accuracy_table = gr.Dataframe(label="Accuracy Summary", interactive=False)

        with gr.Row():
            results_table = gr.Dataframe(label="Detailed Predictions", interactive=False)

        with gr.Row():
            chart_output = gr.Plot(label="Performance Visualization")

        analyze_btn.click(
            fn=run_comparison,
            inputs=[url_annual, url_forest, url_herbaceous, url_highway, url_industrial,
                   url_pasture, url_permanent, url_residential, url_river, url_sealake],
            outputs=[results_table, accuracy_table, chart_output, progress_output]
        )

        gr.Markdown("""
        ---
        ### 📋 Instructions
        1. Paste URLs of satellite/aerial images for each of the 10 categories
        2. Click "Run Comparison" to analyze with all loaded models
        3. View accuracy metrics, detailed predictions, and visualizations
        4. All models must be present in the default paths to load

        ### 🎯 Categories
        - **AnnualCrop**: Land with annual crops
        - **Forest**: Dense tree coverage
        - **HerbaceousVegetation**: Grass, shrubs, non-woody plants
        - **Highway**: Roads and major transportation routes
        - **Industrial**: Industrial zones and facilities
        - **Pasture**: Grassland for livestock
        - **PermanentCrop**: Orchards, vineyards, permanent crops
        - **Residential**: Urban/suburban residential areas
        - **River**: Water bodies and rivers
        - **SeaLake**: Sea and large lakes
        """)

    return demo

if __name__ == "__main__":
    demo = create_ui()
    print("🚀 Launching EuroSAT Multi-Model Comparison Interface...")
    print(f"📊 Loaded Models: {list(MODELS_DICT.keys())}")
    print("\n" + "="*60)
    print("🌐 Gradio Interface Starting...")
    print("="*60 + "\n")
    demo.launch(
        server_name="0.0.0.0",
        server_port=7860,
        share=True,
        show_error=True
    )